In [11]:
import sys
!{sys.executable} --version

# ! export CMAKE_PREFIX_PATH="/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/symengine/0.14.0"
# sys.path.insert(0, "/project/6006115/gjones/molrep/lib/python3.10/site-packages")
# !{sys.executable} -m pip uninstall numpy --yes
# !{sys.executable} -m pip install --no-index --upgrade pip
# !{sys.executable} -m pip install -e /home/gjones/projects/def-jacobsen/gjones/qiskit-addon-dice-solver/
# !{sys.executable} -m pip install numpy==1.26.4
# !{sys.executable} -m pip install -e /scratch/gjones/distributed_LUCJ/
# !pip install -e /home/gjones/projects/def-jacobsen/gjones/qiskit-addon-dice-solver/
# !pip install -e /scratch/gjones/distributed_LUCJ/
# !{sys.executable} -m pip install pandas


import psutil
from functools import partial
 
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd
#import tensorflow as tf
import os
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import pickle
# import xgboost as xgb

from glob import glob
# import psi4
# from helper_CC_ML_spacial import *

import pyscf
from pyscf import gto, scf, mcscf, cc

import ffsim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

from qiskit_addon_sqd.fermion import SCIResult, diagonalize_fermionic_hamiltonian
from qiskit_addon_sqd.counts import bit_array_to_arrays
from qiskit.primitives import StatevectorSampler, BitArray


from ansatzmap import get_zigzag_physical_layout

from tqdm import tqdm

from DDLUCJ import DDLUCJ, GrabAmps    
font_path = 'Futura Book.ttf'
font_manager.fontManager.addfont(font_path)
prop = font_manager.FontProperties(fname=font_path, size='large')
plt.rcParams['font.family'] = prop.get_name()
plt.rcParams.update({'font.size': 12})


Python 3.13.7


In [12]:
UofT_palette = [ "#1E3765",
                 "#007FA3", 
                 "#6D247A", 
                 "#DC4633",
                 "#6FC7EA",
                 "#00A189",
                 "#AB1368",
                 "#0D534D",
                 "#F1C500",
                 "#8DBF2E"
               ]

palette = sns.color_palette(UofT_palette)

In [13]:
datadf = pd.read_csv("../../../DDLUCJ_active_spaces_unfrozen.csv",delimiter=';').dropna(axis=1)

In [14]:
datadf

,molecule,formula,xyz,No,Ne
0,ammonia,NH3,ammonia157.xyz,8,10
1,methane,CH4,methane50.xyz,9,10
2,ethylene,C2H4,ethylene42.xyz,14,16
3,ethane,C2H6,ethane28.xyz,16,18
4,water,H2O,water183.xyz,7,10
5,formaldehyde,CH2O,formaldehyde138.xyz,12,16
6,methanol,CH3OH,methanol22.xyz,14,18
7,fluoroform,CHF3,GDB04_5.xyz,21,34
8,"buta-1,3-diene",C4H6,GDB04_53.xyz,26,30
9,but-1-yne,C4H6,GDB04_49.xyz,26,30


In [15]:
dim_df = pd.read_excel("Dimensions.xlsx",index_col=0)

In [16]:
structure_path_dict = dict(zip(datadf['molecule'], datadf['xyz']))

structure_path_dict = {
    name: os.path.join("../../../classical/structures", xyz)
    for name, xyz in structure_path_dict.items()
}

In [17]:
moldf = pd.read_csv('molecules.csv')
activespacedf = pd.read_csv("active_spaces.csv")

In [18]:
# "No recovery" baseline: a single round of hamming-weight postselection on the
# raw quantum samples, diagonalized directly with Fulqrum -- no configuration
# recovery iterations. See DDLUCJ.PostprocessFulqrum(use_recovery=False).
num_batches = 1
samples_per_batch = 1000  # cap on unique half-strings included in the subspace


In [19]:
energy_data = []
for i in tqdm(sorted(glob("./counts/*npz"))):
    parts = os.path.basename(i).replace('.npz', '').split('_')
    name, _, rawlayers, basis = parts[:4]
    injection = '_'.join(parts[4:])
    layer = int(rawlayers.strip("L"))

    moldict = moldf[moldf['molecule'] == name]
    if moldict.empty:
        print(f"MISSING IN MOLECULES.CSV: {name}")
        continue
    n_electrons = moldict['n_electrons'].values[0]
    num_orbitals = moldict['num_orbitals'].values[0]
    xyzname = moldict['mol_filename'].values[0]
    pathxyz = os.path.join("../../../classical/structures/", xyzname)

    print(f"Running {name}_LUCJ_L{layer}_{basis}_{injection}")

    ampdict = GrabAmps(name, basis)
    t1, t2 = ampdict[injection]

    dd = DDLUCJ(
        StructurePath=pathxyz,
        BasisSet=basis,
        NElec=int(n_electrons),
        NOrb=int(num_orbitals),
        injected=True,
        t1=t1,
        t2=t2,
        n_reps=int(layer),
        optimization_level=3,
        temp_dir="./",
        clean_temp_dir=True,
        n_jobs=-1,
        num_batches=1,          # see note below
        samples_per_batch=1000,
        verbose=True,
    )

    counts = np.load(i)
    bitstrings = BitArray.from_bool_array(counts['bitstrings'])

    energy, subspace = dd(
        postprocess=True,
        BitArray=bitstrings,
        usefulqrum=True,
        bitarraypath=i,
        use_recovery=False,
    )
    energy = float(np.ravel(energy)[0])

    energy_data.append((name, layer, basis, injection, energy, subspace))

  0%|                                                                                                                                                                                                                | 0/1080 [00:00<?, ?it/s]

Running (Z)-1-fluoroprop-1-ene_LUCJ_L1_STO-3G_CCSD
Active Space Orbitals: 25, Electrons: 32, Frozen: 0
Active indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
use_recovery=False: overriding max_iterations=100 -> 1
ITERATION 0
  num total half strs:  267
  num selected half strs:  267
  Half strs construction took: 0.0001 seconds
  Subspace construction took: 0.013388 seconds
  Subspace dimension: 267 x 267 = 71_289


  0%|▏                                                                                                                                                                                                    | 1/1080 [00:37<11:09:33, 37.23s/it]

  Operator projection took: 5.810299 seconds
  CSR matrix memory: 2.730381 MBs
  Initial guess vector v0 construction took: 0.0002 seconds
  Starting eigensolving ...
  Eigensolving took: 0.0145 seconds
  Electronic Energy: [-329.05716361]
  Total Energy: [-213.11648488]
  num carryover full strs: 6
Iter 0 took: 5.8551 seconds

Running (Z)-1-fluoroprop-1-ene_LUCJ_L1_STO-3G_ML
Active Space Orbitals: 25, Electrons: 32, Frozen: 0
Active indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
use_recovery=False: overriding max_iterations=100 -> 1
ITERATION 0
  num total half strs:  173
  num selected half strs:  173
  Half strs construction took: 0.0001 seconds
  Subspace construction took: 0.005869 seconds
  Subspace dimension: 173 x 173 = 29_929


  0%|▎                                                                                                                                                                                                    | 2/1080 [01:10<10:27:20, 34.92s/it]

...

 52%|████████████████████████████████████████████████████████████████████████████████████████████████████▏                                                                                             | 558/1080 [4:31:40<1:39:41, 11.46s/it]

  Operator projection took: 0.367299 seconds
  CSR matrix memory: 3.732426 MBs
  Initial guess vector v0 construction took: 0.0000 seconds
  Starting eigensolving ...
  Eigensolving took: 0.0039 seconds
  Electronic Energy: [-469.17852899]
  Total Energy: [-336.79066847]
  num carryover full strs: 263
Iter 0 took: 0.3767 seconds

Running fluoroform_LUCJ_L2_STO-3G_CCSD
Active Space Orbitals: 21, Electrons: 34, Frozen: 0
Active indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


 52%|████████████████████████████████████████████████████████████████████████████████████████████████████▏                                                                                             | 558/1080 [4:31:51<4:14:19, 29.23s/it]

use_recovery=False: overriding max_iterations=100 -> 1
ITERATION 0


ValueError: samples_per_batch (= 0) must > 0

In [20]:
# sanity check on the last file processed
name, layers, basis, injection, energy, subspace_dimension = energy_data[-1]
print(f"{name} (L={layers}, {basis}, {injection}): E = {energy:.6f} Ha, subspace dim = {subspace_dimension}")


fluoroform (L=1, cc-pVDZ, zeroes): E = -336.790668 Ha, subspace dim = 4761


In [21]:
norecovery_df = pd.DataFrame(
    energy_data,
    columns=['Name', 'L', 'Basis', 'Injection', 'Energy_NoRecovery', 'SubspaceDim_NoRecovery'],
)
norecovery_df.to_excel("Energies_NoRecovery.xlsx")
norecovery_df


,Name,L,Basis,Injection,Energy_NoRecovery,SubspaceDim_NoRecovery
0,(Z)-1-fluoroprop-1-ene,1,STO-3G,CCSD,-213.116485,71289
1,(Z)-1-fluoroprop-1-ene,1,STO-3G,ML,-213.116475,29929
2,(Z)-1-fluoroprop-1-ene,1,STO-3G,ML_exact,-213.116469,39204
3,(Z)-1-fluoroprop-1-ene,1,STO-3G,MP2,-213.116469,5929
4,(Z)-1-fluoroprop-1-ene,1,STO-3G,random,-213.116469,25921
...,...,...,...,...,...,...
553,fluoroform,1,cc-pVDZ,ML,-336.786784,9
554,fluoroform,1,cc-pVDZ,ML_exact,-336.786784,25
555,fluoroform,1,cc-pVDZ,MP2,-336.786784,81
556,fluoroform,1,cc-pVDZ,random,-336.786784,49
